# Environment

In [1]:
import os
os.chdir("..")

In [2]:
import pandas as pd
import numpy as np
import spacy

from src.utils.path import DATA_PATH

In [3]:
nlp = spacy.load("pt_core_news_lg")

# Load Lexicons

In [4]:
liwc = pd.read_csv(f"{DATA_PATH}/lexicons/liwc.csv")
sentilex = pd.read_csv(f"{DATA_PATH}/lexicons/sentilex.csv")
wordnetaffect = pd.read_csv(f"{DATA_PATH}/lexicons/wordnetaffect.csv")

In [5]:
lex = pd.concat([liwc, sentilex, wordnetaffect]).sort_values(by='polarity').drop_duplicates(subset=['word'], keep='first').reset_index(drop=True)

## Filter Polarities

In [6]:
pos_set = set(lex.loc[lex.polarity == 1, 'word'].str.lower())
neg_set = set(lex.loc[lex.polarity == -1, 'word'].str.lower())
nto_set = set(lex.loc[lex.polarity == 0, 'word'].str.lower())

# Feature Extraction Functions

In [7]:
def extract_lex_features(text):
    doc = nlp(text)
    toks_lower = [token.text.lower() for token in doc if not token.is_stop]
    toks_lemma_lower = [token.lemma_.lower() for token in doc if not token.is_stop]
    n = max(len(toks_lower), 1)

    pos_tokens = [t for t in toks_lower if t in pos_set] + [t for t in toks_lemma_lower if t in pos_set]
    pos_count = len(pos_tokens)

    neg_tokens = [t for t in toks_lower if t in neg_set] + [t for t in toks_lemma_lower if t in neg_set]
    neg_count = len(neg_tokens)

    nto_tokens = [t for t in toks_lower if t in nto_set] + [t for t in toks_lemma_lower if t in nto_set]
    nto_count = len(nto_tokens)
    
    return {
        "feat_pos_ratio": round(pos_count/n, 2),
        "feat_neg_ratio": round(neg_count/n, 2),
        "feat_nto_ratio": round(nto_count/n, 2),
    }

# Feature Engineering in Corpora

## Hate-BR

In [8]:
hate = pd.read_csv(f"{DATA_PATH}/hate-br/processed/hate-br.csv")

In [9]:
hate_features = hate['text'].apply(extract_lex_features).to_list()
hate_features = pd.DataFrame(hate_features)

In [10]:
hate_features.to_csv(f"{DATA_PATH}/hate-br/features/lex.csv", index=False)

## Ited-BR

In [11]:
ited = pd.read_csv(f"{DATA_PATH}/ited-br/processed/ited-br.csv")

In [12]:
ited_features = ited['text'].apply(extract_lex_features).to_list()
ited_features = pd.DataFrame(ited_features)

In [13]:
ited_features.to_csv(f"{DATA_PATH}/ited-br/features/lex.csv", index=False)

## Unified-Hate

In [14]:
unified = pd.read_csv(f"{DATA_PATH}/unified-hate/processed/unified-hate.csv")

In [15]:
unified_features = unified['text'].apply(extract_lex_features).to_list()
unified_features = pd.DataFrame(unified_features)

In [16]:
unified_features.to_csv(f"{DATA_PATH}/unified-hate/features/lex.csv", index=False)